In [ ]:
# D600 Statistical Data Mining
# Task 1

#Importing Libraries
import pandas as pd # pandas for data manipulation and analysis
import seaborn as sns # seaborn for data visualization
import matplotlib.pyplot as plt # matplotlib for data visualization
import statsmodels.api as sm # statsmodels for statistical modeling
from sklearn.linear_model import LinearRegression # sklearn for machine learning
from sklearn.model_selection import train_test_split # sklearn for splitting data into training and testing sets
from sklearn.metrics import mean_squared_error # sklearn for evaluating model performance
from statsmodels.stats.outliers_influence import variance_inflation_factor # statsmodels for calculating VIF

file_path = r"../data/housing_data.csv"

df = pd.read_csv(file_path)

print(df.columns)

In [ ]:
df

In [ ]:
#Descriptive statistics
variable_columns = ['Price', 'SquareFootage', 'SchoolRating']
stats = df[variable_columns].describe()

# Display the results
display(stats)

In [ ]:
# Univariate visualizations

# Histogram of Price distribution
plt.figure(figsize=(8, 6))
sns.histplot(df['Price'], kde=True, bins=10, color='skyblue', edgecolor='black')
plt.title('Distribution of Home Prices', fontsize=16, fontweight='bold', color='darkblue')
plt.xlabel('Price', fontsize=12, fontweight='bold', color='darkblue')
plt.ylabel('Frequency', fontsize=12, fontweight='bold', color='darkblue')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# Histogram of square Footage distribution
plt.figure(figsize=(8, 6))
sns.histplot(df['SquareFootage'], kde=True, bins=10, color='lightgreen', edgecolor='black')
plt.title('Distribution of Square Footage', fontsize=16, fontweight='bold', color='darkgreen')
plt.xlabel('Square Footage', fontsize=12, fontweight='bold', color='darkgreen')
plt.ylabel('Frequency', fontsize=12, fontweight='bold', color='darkgreen')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# Histogram of school rating distribution
plt.figure(figsize=(8, 6))
sns.histplot(df['SchoolRating'], kde=True, bins=10, color='salmon', edgecolor='black')
plt.title('Distribution of School Ratings', fontsize=16, fontweight='bold', color='darkred')
plt.xlabel('School Ratings', fontsize=12, fontweight='bold', color='darkred')
plt.ylabel('Frequency', fontsize=12, fontweight='bold', color='darkred')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# Bivariate visualizations

# Scatter plot: Price vs. Square Footage
plt.figure(figsize=(8, 6))
sns.scatterplot(x='SquareFootage', y='Price', data=df, color='darkorange', s=50, edgecolor='black', alpha=0.8)
plt.title('Price vs Square Footage', fontsize=16, fontweight='bold', color='darkblue')
plt.xlabel('Square Footage', fontsize=12, fontweight='bold', color='darkblue')
plt.ylabel('Price', fontsize=12, fontweight='bold', color='darkblue')
plt.grid(axis='both', linestyle='--', alpha=0.5)
plt.show()

# Scatter plot: Price vs. School Rating
plt.figure(figsize=(8, 6))
sns.scatterplot(x='SchoolRating', y='Price', data=df, color='purple', s=50, edgecolor='black', alpha=0.8)
plt.title('Price vs School Rating', fontsize=16, fontweight='bold', color='darkred')
plt.xlabel('School Rating', fontsize=12, fontweight='bold', color='darkred')
plt.ylabel('Price', fontsize=12, fontweight='bold', color='darkred')
plt.grid(axis='both', linestyle='--', alpha=0.5)
plt.show()

In [ ]:
#  Split the data into two datasets, with a larger percentage assigned to the training dataset and a smaller percentage assigned to the test data set. Provide the files.

# Only use the relevant variables
variables = ['Price', 'SquareFootage', 'SchoolRating']
df_subset = df[variables]

# Split into train (80%) and test (20%) sets
train_df, test_df = train_test_split(df_subset, test_size=0.2, random_state=42)

# Save as CSV
train_df.to_csv("train_data.csv", index=False)
test_df.to_csv("test_data.csv", index=False)


In [ ]:
#  Use the training dataset to create and perform a regression model using regression as a statistical method.

# Define X and y from the training data
X_train = train_df[['SquareFootage', 'SchoolRating']]
y_train = train_df['Price']

# Add constant for intercept
X_train_const = sm.add_constant(X_train)

# Fit the regression model
model = sm.OLS(y_train, X_train_const).fit()

# Show full summary
model_summary = model.summary()
print(model_summary)


In [ ]:
# Optimize Model

# Start with both variables
X_full = train_df[['SquareFootage', 'SchoolRating']]
y = train_df['Price']
X_full_const = sm.add_constant(X_full)
full_model = sm.OLS(y, X_full_const).fit()

# Stepwise check: Look at p-values
print(full_model.summary())

# For example, let's say 'SchoolRating' has p > 0.05
X_reduced = train_df[['SquareFootage']]
X_reduced_const = sm.add_constant(X_reduced)
reduced_model = sm.OLS(y, X_reduced_const).fit()

# Compare both models
print("\nReduced Model:")
print(reduced_model.summary())

In [ ]:
# Mean Squared Error (MSE)

# Predict on training data using the optimized model
y_train_pred = full_model.predict(X_full_const)

# Calculate MSE
mse = mean_squared_error(y, y_train_pred)
print("Mean Squared Error (MSE) on training set:", mse)

In [ ]:
# Model Accuracy

# Prepare test data
X_test = test_df[['SquareFootage', 'SchoolRating']]
y_test = test_df['Price']
X_test_const = sm.add_constant(X_test)

# Make predictions using the trained (optimized) model
y_test_pred = full_model.predict(X_test_const)

# Calculate MSE on test data
mse_test = mean_squared_error(y_test, y_test_pred)
print("Mean Squared Error (MSE) on test set:", mse_test)

In [ ]:
# 1. Linearity + Homoscedasticity (plot residuals)
residuals = full_model.resid
fitted = full_model.fittedvalues

plt.figure(figsize=(8,6))
sns.scatterplot(x=fitted, y=residuals)
plt.axhline(0, linestyle='--', color='red')
plt.title("Residuals vs. Fitted Values")
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.show()

# 2. Normality of Residuals
sm.qqplot(residuals, line='s')
plt.title("Q-Q Plot of Residuals")
plt.show()

# 3. Multicollinearity (VIF)
X = train_df[['SquareFootage', 'SchoolRating']]
X_const = sm.add_constant(X)

vif_data = pd.DataFrame()
vif_data["feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X_const.values, i+1) for i in range(X.shape[1])]
print(vif_data)
